# 1. Install & Import Libraries

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install transformers scikit-learn pillow pandas numpy matplotlib huggingface_hub ipywidgets opencv-python google-cloud-storage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import ConvNextForImageClassification, AutoImageProcessor
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from google.cloud import storage
from torch.amp import autocast

# 2. Ensemble Strategy & Paths Configuration
Toggle your strategies here as defined in the `ensemble_strategies.md` guide.

In [ ]:
# ============================================================
# ENSEMBLE CONFIGURATION FLAGS
# ============================================================
ENSEMBLE_FLAGS = {
    "use_tta": True,
    "strategy": "simple_avg", # Options: simple_avg, weighted_avg, per_class_weighted, majority_vote, rank_fusion, all
    "convnext_weight": 0.55,
    "efficientnet_weight": 0.45,
    "per_class_weights": {
        "convnext":     [0.40, 0.65, 0.50, 0.60, 0.55],
        "efficientnet": [0.60, 0.35, 0.50, 0.40, 0.45],
    }
}

# Checkpoint Directories
# Using Exp 5 for EfficientNet (CE+LS+MixUp+ECA+DLR)
EFFNET_DIR = "../exp_5_efficienetb3_CE+Label_Smoothing+MixUp+ECA+Disc.LR+TTA/"
CONVNEXT_DIR = "../../convnext_placeholder_dir/" # Replace with actual path when available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cuda available: ", torch.cuda.is_available())

# 3. Dataset Download & Prep

In [ ]:
client = storage.Client()
bucket = client.bucket('naic-dataset-images')
os.makedirs('dataset/image', exist_ok=True)

print("Fetching labels.csv...")
blob_csv = bucket.blob('labels.csv')
blob_csv.download_to_filename('dataset/labels.csv')

# 3. Download the images
blobs = bucket.list_blobs(prefix='image/')
print("\nStarting image download...")

download_count = 0
for i, blob in enumerate(blobs):
    if not blob.name.endswith('/'):
        filename = blob.name.split('/')[-1]
        local_path = f'dataset/image/{filename}'

        # Only download if the file isn't already on the disk
        if not os.path.exists(local_path):
            blob.download_to_filename(local_path)
            download_count += 1

    if i % 500 == 0:
        print(f"Scanned {i} files...")

print(f"Download complete! Fetched {download_count} new images.")

# 4. Read Data & Test Split

In [ ]:
df = pd.read_csv('dataset/labels.csv')
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['Label'],
    random_state=42
)
print(f"Test set size: {len(test_df)}")

# 5. Preprocessing (CLAHE + Crop)

In [ ]:
def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        check_shape = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_shape == 0:
            return img
        else:
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
        return img
    return img

def apply_clahe_green_channel(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    r, g, b = cv2.split(image)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    g_clahe = clahe.apply(g)
    return cv2.merge((r, g_clahe, b))

def preprocess_fundus(image_pil):
    img_cv = np.array(image_pil)
    img_cropped = crop_image_from_gray(img_cv)
    img_clahe = apply_clahe_green_channel(img_cropped)
    return Image.fromarray(img_clahe)

# 6. Dataset Loader

In [ ]:
class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None, use_preprocessing=True):
        self.data = df
        self.img_dir = img_dir
        self.transform = transform
        self.use_preprocessing = use_preprocessing

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image']
        label = int(self.data.iloc[idx]['Label'])
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        if self.use_preprocessing:
            image = preprocess_fundus(image)
        if self.transform:
            image = self.transform(image)
            
        return image, label

# 7. Attention Modules (ECA)

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention with adaptive kernel size."""
    def __init__(self, in_channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((np.log2(in_channels)) / gamma) + b / gamma)
        k = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(1, 2)
        y = self.conv(y)
        y = self.sigmoid(y)
        y = y.transpose(1, 2).unsqueeze(-1)
        return x * y

# 8. Model Architectures

In [ ]:
import torchvision.models as models

class EfficientNetWithECA(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        # Load standard EffNetB3 architecture
        base = models.efficientnet_b3(weights=None)
        self.features = base.features
        
        # The last feature dimension for EffNetB3 is 1536
        feature_dim = 1536
        self.eca = ECA(feature_dim)
        self.avgpool = base.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(p=base.classifier[0].p, inplace=True),
            nn.Linear(feature_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.eca(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

class ConvNextWithECA(nn.Module):
    def __init__(self, base_model, feature_dim):
        super().__init__()
        self.convnext = base_model.convnext
        self.eca = ECA(feature_dim)
        self.classifier = base_model.classifier
        
    def forward(self, pixel_values):
        features = self.convnext(pixel_values).last_hidden_state
        attended = self.eca(features)
        pooled = attended.mean(dim=[-2, -1])
        return self.classifier(pooled)

# 9. Model Loading & TTA Transforms

In [ ]:
# Transforms for EfficientNet (300x300) and ConvNeXt (224x224)
effnet_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

convnext_processor = AutoImageProcessor.from_pretrained('facebook/convnext-small-224')
convnext_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=convnext_processor.image_mean, std=convnext_processor.image_std)
])

def get_tta_transforms(base_size, processor_mean, processor_std):
    return [
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomRotation(degrees=(10, 10)), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomRotation(degrees=(-10, -10)), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.ColorJitter(brightness=0.15), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomHorizontalFlip(p=1.0), transforms.RandomRotation(degrees=(10, 10)), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
        transforms.Compose([transforms.Resize((base_size, base_size)), transforms.RandomVerticalFlip(p=1.0), transforms.RandomRotation(degrees=(-10, -10)), transforms.ToTensor(), transforms.Normalize(mean=processor_mean, std=processor_std)]),
    ]

effnet_tta = get_tta_transforms(300, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
convnext_tta = get_tta_transforms(224, convnext_processor.image_mean, convnext_processor.image_std)

# 10. Inference Engine

In [ ]:
def run_inference_for_model(model_cls, model_args, weights_dir, test_df, tta_transforms,
                            is_huggingface=False, weight_pattern="best_fold_{fold}.pth",
                            num_folds=5):
    """Loads all folds, runs TTA inference, averages across TTA views and folds.
    Returns probabilities of shape (N_samples, 5)."""
    ensemble_probs = []
    loaded_folds = []

    for fold in range(num_folds):
        model_path = os.path.join(weights_dir, weight_pattern.format(fold=fold))
        if not os.path.exists(model_path):
            print(f"  [fold {fold}] SKIP — not found: {model_path}")
            continue

        print(f"  [fold {fold}] loading {os.path.basename(model_path)}")

        # Initialize architecture
        if is_huggingface:
            base = ConvNextForImageClassification.from_pretrained(
                'facebook/convnext-small-224', num_labels=5, ignore_mismatched_sizes=True
            )
            model = model_cls(base, **model_args).to(device)
        else:
            model = model_cls(**model_args).to(device)

        # Accept either a plain state_dict or a checkpoint dict containing one
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif isinstance(ckpt, dict) and "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        else:
            state_dict = ckpt
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing:
            print(f"    missing keys: {len(missing)} (showing 3): {missing[:3]}")
        if unexpected:
            print(f"    unexpected keys: {len(unexpected)} (showing 3): {unexpected[:3]}")
        model.eval()

        # TTA loop
        transforms_to_run = tta_transforms if ENSEMBLE_FLAGS["use_tta"] else [tta_transforms[0]]
        tta_probs_per_view = []

        for tta_transform in transforms_to_run:
            tta_dataset = MedicalDatasetLoader(test_df, "dataset/image", tta_transform, use_preprocessing=True)
            tta_loader = DataLoader(tta_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

            view_probs = []
            with torch.no_grad():
                for inputs, _ in tta_loader:
                    inputs = inputs.to(device)
                    with autocast(device_type='cuda'):
                        outputs = model(inputs)
                    probs = F.softmax(outputs.float(), dim=1)
                    view_probs.append(probs.cpu())
            tta_probs_per_view.append(torch.cat(view_probs, dim=0))

        # Average across TTA views for this fold
        avg_fold_probs = torch.stack(tta_probs_per_view).mean(dim=0)
        ensemble_probs.append(avg_fold_probs)
        loaded_folds.append(fold)

        # Free GPU mem before next fold
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not ensemble_probs:
        print("  No models loaded — returning random probabilities so the pipeline can be tested.")
        return F.softmax(torch.randn(len(test_df), 5), dim=1)

    print(f"  Averaging across {len(loaded_folds)} fold(s): {loaded_folds}")
    return torch.stack(ensemble_probs).mean(dim=0)


print("=== Running EfficientNet Inference (5 folds: best_fold_0.pth … best_fold_4.pth) ===")
effnet_probs = run_inference_for_model(
    model_cls=EfficientNetWithECA,
    model_args={"num_classes": 5},
    weights_dir=EFFNET_DIR,
    test_df=test_df,
    tta_transforms=effnet_tta,
    is_huggingface=False,
    weight_pattern="best_fold_{fold}.pth",
    num_folds=5,
)

print("\n=== Running ConvNeXt Inference (5 folds: best_fold_0.pth … best_fold_4.pth) ===")
convnext_probs = run_inference_for_model(
    model_cls=ConvNextWithECA,
    model_args={"feature_dim": 768},
    weights_dir=CONVNEXT_DIR,
    test_df=test_df,
    tta_transforms=convnext_tta,
    is_huggingface=True,
    weight_pattern="best_fold_{fold}.pth",
    num_folds=5,
)

# 11. Ensemble Strategies

In [ ]:
def evaluate_predictions(probs, true_labels, strategy_name):
    preds = probs.argmax(dim=1).numpy()
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, average='macro', zero_division=0)
    rec = recall_score(true_labels, preds, average='macro', zero_division=0)
    f1 = f1_score(true_labels, preds, average='macro')
    try:
        roc_auc = roc_auc_score(true_labels, probs.numpy(), multi_class='ovr', average='macro')
    except ValueError:
        roc_auc = 0.0
        
    print(f"\n{'='*40}")
    print(f"Strategy: {strategy_name}")
    print(f"{'='*40}")
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall:    {rec * 100:.2f}%")
    print(f"F1 Score:  {f1 * 100:.2f}%")
    print(f"ROC-AUC:   {roc_auc * 100:.2f}%")
    print("\nConfusion Matrix:")
    print(confusion_matrix(true_labels, preds))
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": roc_auc}

true_labels = test_df['Label'].values
results = {}

def strategy_simple_avg():
    probs = 0.5 * effnet_probs + 0.5 * convnext_probs
    return evaluate_predictions(probs, true_labels, "Simple Average")

def strategy_weighted_avg():
    w_c = ENSEMBLE_FLAGS["convnext_weight"]
    w_e = ENSEMBLE_FLAGS["efficientnet_weight"]
    probs = w_e * effnet_probs + w_c * convnext_probs
    return evaluate_predictions(probs, true_labels, f"Weighted Average ({w_c} Conv, {w_e} Eff)")

def strategy_per_class_weighted():
    w_c = torch.tensor(ENSEMBLE_FLAGS["per_class_weights"]["convnext"])[None, :]
    w_e = torch.tensor(ENSEMBLE_FLAGS["per_class_weights"]["efficientnet"])[None, :]
    probs = w_e * effnet_probs + w_c * convnext_probs
    # Normalize probabilities so they sum to 1
    probs = probs / probs.sum(dim=1, keepdim=True)
    return evaluate_predictions(probs, true_labels, "Per-Class Weighted")

def strategy_majority_vote():
    eff_preds = effnet_probs.argmax(dim=1)
    conv_preds = convnext_probs.argmax(dim=1)
    
    final_preds = []
    for e, c in zip(eff_preds, conv_preds):
        if e == c:
            final_preds.append(e.item())
        else:
            # Tie breaker: ConvNeXt
            final_preds.append(c.item())
            
    # Convert to one-hot for AUC calculation compatibility in the eval function
    probs = torch.zeros_like(convnext_probs)
    probs.scatter_(1, torch.tensor(final_preds).unsqueeze(1), 1.0)
    return evaluate_predictions(probs, true_labels, "Majority Voting (Tiebreaker: ConvNeXt)")

def strategy_rank_fusion():
    # Rank probabilities: 0 is lowest prob, 4 is highest prob
    eff_ranks = effnet_probs.argsort(dim=1).argsort(dim=1)
    conv_ranks = convnext_probs.argsort(dim=1).argsort(dim=1)
    
    # Sum the ranks (higher sum means better)
    summed_ranks = eff_ranks + conv_ranks
    probs = summed_ranks.float() / summed_ranks.sum(dim=1, keepdim=True)
    return evaluate_predictions(probs, true_labels, "Borda Rank Fusion")

# 12. Run Strategy

In [ ]:
strategy_map = {
    "simple_avg": strategy_simple_avg,
    "weighted_avg": strategy_weighted_avg,
    "per_class_weighted": strategy_per_class_weighted,
    "majority_vote": strategy_majority_vote,
    "rank_fusion": strategy_rank_fusion
}

if ENSEMBLE_FLAGS["strategy"] == "all":
    for name, func in strategy_map.items():
        results[name] = func()
else:
    selected = ENSEMBLE_FLAGS["strategy"]
    results[selected] = strategy_map[selected]()

# 13. Strategy Comparison Summary

In [ ]:
if ENSEMBLE_FLAGS["strategy"] == "all" or len(results) > 1:
    print("\n" + "*"*60)
    print("                 ENSEMBLE STRATEGY COMPARISON")
    print("*"*60)
    print(f"{'Strategy':<25} | {'Accuracy':<8} | {'Recall':<8} | {'F1':<8}")
    print("-"*60)
    for name, res in results.items():
        print(f"{name:<25} | {res['Accuracy']*100:>7.2f}% | {res['Recall']*100:>7.2f}% | {res['F1']*100:>7.2f}%")
    print("*"*60)
else:
    print("Set ENSEMBLE_FLAGS['strategy'] = 'all' to compare all strategies side-by-side.")